# Sillon : comparer des variantes du diagnostic photo

Ce notebook entraîne en une passe plusieurs variantes du petit modèle qui reconnaît, sur le téléphone et sans Internet,
les maladies et ravageurs du potager, puis compare leurs résultats **sur les photos de terrain jamais vues** :

| Variante | Réseau | Image | Recette | Mildiou + alternariose |
|---|---|---|---|---|
| `baseline_small224` | MobileNetV3-Small (le modèle actuel) | 224 px | historique | séparés |
| `large288_improved` | MobileNetV3-Large | 288 px | améliorée (mixup/cutmix, dégel progressif, cosine) | séparés |
| `large288_improved_merge` | MobileNetV3-Large | 288 px | améliorée | fusionnés en `tache_feuille` |

**Avant de lancer** (voir aussi `tools/ml/README.md`, procédure pas à pas) :
1. Compte Kaggle avec **téléphone vérifié** (sinon pas de GPU ni d'Internet dans les notebooks).
2. **Settings > Accelerator : GPU T4 x2 ou P100**, **Internet : On**.
3. **Add Data** : les jeux de la cellule de configuration ci-dessous (noms à renseigner, voir les TODO).
4. Remplissez la cellule de configuration, puis « Run All », ou mieux **Save Version > Save & Run All (Commit)** :
   le calcul continue sans que l'onglet reste ouvert et la sortie est conservée.

**Durée estimée (estimation, non mesurée : à corriger après le premier passage)** : baseline 30 à 60 min,
chaque variante Large 288 px 2 à 4 h sur T4, soit environ 5 à 9 h pour les trois. Le quota Kaggle GPU est
d'environ 30 h par semaine et une session est limitée à quelques heures (9 à 12 h, à vérifier dans l'interface).

**Reprise** : chaque variante écrit ses points de reprise dans `out_<variante>/checkpoints` (un par palier d'entraînement).
Une variante terminée n'est jamais relancée ; une variante interrompue repart du dernier palier terminé (`--resume`).
Si la session a été coupée, ajoutez la sortie de la version précédente du notebook comme jeu de données
(**Add Data > Notebook Output**) et renseignez `PREVIOUS_OUTPUT` plus bas.

In [ ]:
# ---- Configuration : à renseigner avant « Run All » -------------------------------------------------
# Les jeux de données sont repérés par un fragment du nom de leur dossier dans /kaggle/input.
# Ne mettez que des noms vérifiés sur la page Kaggle du jeu ; tant que ce n'est pas fait, laissez TODO.
PLANTVILLAGE_HINT = "village"   # fragment du dossier PlantVillage (photos de labo) ; à confirmer après Add Data
PLANTDOC_HINT = "doc"           # fragment du dossier PlantDoc (photos de terrain) ; à confirmer après Add Data

# Photos de terrain par classe Sillon (sortie de collect_data.py, avec ATTRIBUTIONS.csv).
# Soit un jeu Kaggle que vous avez créé (fragment du nom), soit None : le notebook lance collect_data.py
# (Internet On, environ 20 min) et utilise le résultat.
CLASSED_HINT = None             # TODO : nom de votre jeu iNaturalist déjà collecté, sinon None

# Jeux supplémentaires de photos de terrain (optionnels), au format dossier-par-classe Sillon
# avec un ATTRIBUTIONS.csv (colonnes path, license). Licences acceptées : CC0, CC-BY, CC-BY-SA.
# Exemple de forme : EXTRA_FIELD_HINTS = ["nom-du-jeu-kaggle"]. Rien n'est prévu tant que la licence n'est pas confirmée.
EXTRA_FIELD_HINTS = []          # TODO : à remplir seulement avec des jeux dont la licence est vérifiée

# Collecte automatique GBIF supplémentaire (optionnel, Internet On)
# Enrichit le modèle avec des données de terrain scientifiques (mildiou, alternariose, rouille, feuillages sains diversifiés).
COLLECT_GBIF_EXTRA = False      # Mettre à True pour collecter automatiquement via expand_dataset.py

# Sortie d'une version précédente du notebook (pour reprendre un calcul coupé), sinon None.
PREVIOUS_OUTPUT = None          # TODO : fragment du nom du jeu « Notebook Output », ou None

# Scripts : dépôt Git public, ou jeu Kaggle contenant le dossier tools/ml (si le dépôt est privé).
REPO_URL = "https://github.com/Relthdrarxash/sillon.git"
SCRIPTS_HINT = None             # TODO : fragment du jeu Kaggle contenant train_sillon.py, ou None pour cloner REPO_URL

# Variantes à exécuter (retirez celles que vous ne voulez pas : le quota GPU est limité).
RUN = ["baseline_small224", "large288_improved", "large288_improved_merge"]

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")


def run(cmd, cwd=None):
    """Lance une commande en affichant sa sortie au fil de l'eau ; renvoie le code de sortie."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    proc = subprocess.Popen([str(c) for c in cmd], cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()


def find_input(hint):
    """Dossier de /kaggle/input dont le nom contient hint (sinon None)."""
    if not hint:
        return None
    found = [p for p in INPUT.iterdir() if p.is_dir() and hint.lower() in p.name.lower()]
    if len(found) > 1:
        print(f"Plusieurs dossiers pour « {hint} » : {[p.name for p in found]} ; le premier est utilisé.")
    return found[0] if found else None


# Scripts d'entraînement
if SCRIPTS_HINT:
    found = find_input(SCRIPTS_HINT)
    assert found, f"Jeu de scripts introuvable pour « {SCRIPTS_HINT} »"
    ML = next((p.parent for p in found.rglob("train_sillon.py")), None)
    assert ML, "train_sillon.py absent du jeu de scripts"
else:
    if not (WORK / "sillon").exists():
        assert run(["git", "clone", "--depth", "1", REPO_URL, WORK / "sillon"]) == 0, "git clone impossible (dépôt privé ? voir SCRIPTS_HINT)"
    ML = WORK / "sillon" / "tools" / "ml"
print("Scripts :", ML)

# Reprise : récupère les sorties d'une version précédente sans écraser ce qui existe déjà
prev = find_input(PREVIOUS_OUTPUT)
if prev:
    for d in prev.rglob("out_*"):
        if d.is_dir() and not (WORK / d.name).exists():
            shutil.copytree(d, WORK / d.name)
            print("Repris de la version précédente :", d.name)

# Jeux de données
inputs = [p.name for p in INPUT.iterdir() if p.is_dir()]
print("Jeux disponibles dans /kaggle/input :", inputs)
plantvillage = find_input(PLANTVILLAGE_HINT)
plantdoc = find_input(PLANTDOC_HINT)
assert plantdoc, "Ajoutez un jeu PlantDoc : sans photos de terrain, le modèle ne vaut rien au jardin."
classed = find_input(CLASSED_HINT)
if not classed:
    classed = WORK / "dataset_raw"
    if not (classed / "ATTRIBUTIONS.csv").exists():
        assert run([sys.executable, "collect_data.py", "--out", classed], cwd=ML) == 0, "collect_data.py a échoué (Internet On ?)"
extra = [find_input(h) for h in EXTRA_FIELD_HINTS]
assert all(extra), f"Jeu supplémentaire introuvable : {EXTRA_FIELD_HINTS}"
if COLLECT_GBIF_EXTRA:
    gbif_dir = WORK / "extra_field_gbif"
    if not (gbif_dir / "ATTRIBUTIONS.csv").exists():
        assert run([sys.executable, "expand_dataset.py", "--gbif", "--out", gbif_dir, "--per-class", "120"], cwd=ML) == 0, "expand_dataset.py a échoué (Internet On ?)"
    extra.append(gbif_dir)
print("PlantVillage :", plantvillage, "| PlantDoc :", plantdoc, "| classes Sillon :", classed, "| extra :", extra)

DATA_ARGS = []
if plantvillage:
    DATA_ARGS += ["--plantvillage", plantvillage]
DATA_ARGS += ["--plantdoc", plantdoc, "--classed", classed]
for p in extra:
    DATA_ARGS += ["--extra-field", p]

In [ ]:
VARIANTS = {
    "baseline_small224": ["--backbone", "small", "--img-size", "224"],
    "large288_improved": ["--backbone", "large", "--img-size", "288", "--recipe", "improved"],
    "large288_improved_merge": ["--backbone", "large", "--img-size", "288", "--recipe", "improved", "--merge-leaf-spots"],
}

for name in RUN:
    out = WORK / f"out_{name}"
    if (out / "sillon_disease.json").exists():
        print(f"== {name} : déjà terminé, ignoré ==")
        continue
    print(f"\n== {name} ==", flush=True)
    code = run([sys.executable, "train_sillon.py", *DATA_ARGS, *VARIANTS[name], "--out", out, "--resume"], cwd=ML)
    if code != 0:
        print(f"{name} : échec (code {code}). Relancez la cellule : la reprise part du dernier palier terminé.")
        break
    # Allège la sortie : le modèle exporté suffit, les points de reprise ne servent plus.
    shutil.rmtree(out / "checkpoints", ignore_errors=True)
    shutil.rmtree(out / "saved_model", ignore_errors=True)
    shutil.rmtree(out / "synthetic", ignore_errors=True)

## Lire le tableau comparatif

Tout vient de `metrics.field_test` (photos de terrain jamais vues, découpage par observateur identique dans toutes les variantes).
- **macro-F1 (12 cl.)** : pour comparer une variante fusionnée à une variante non fusionnée, la variante non fusionnée est
  évaluée aussi avec mildiou et alternariose regroupés (`merged_view`). **macro-F1 natif** : sur les classes du modèle lui-même.
- **rappel mildiou / alternariose** : « exact » = la bonne maladie est nommée ; « groupe » = la photo est rangée parmi les
  taches de feuille (seule mesure possible pour le modèle fusionné). Entre crochets : intervalle de confiance de Wilson à 95 %.
- **couverture** : part des photos pour lesquelles l'appli ose répondre au seuil retenu (justesse visée : 85 %).
- Colonne `TTA` : même modèle évalué avec 5 vues par photo. L'appli ne fait **pas** de TTA : c'est la colonne « simple » qui compte.
- Un intervalle large (classe de moins de 60 photos de test) ne permet pas de conclure.

In [ ]:
def pct(x):
    return f"{100 * x:.0f} %"


def with_ci(value, ci):
    return f"{pct(value)} [{pct(ci[0])}-{pct(ci[1])}]"


def summarize(meta, key):
    """Chiffres clés d'une variante, évaluation simple (field_test) ou TTA (field_test_tta)."""
    block = meta["metrics"][key]
    merged = meta.get("merge_leaf_spots", False)
    comparable = block["macro_f1"] if merged else block.get("merged_view", {}).get("macro_f1", block["macro_f1"])
    group = block["leaf_spot_group"]["members"]
    row = {"macro-F1 (12 cl.)": f"{comparable:.3f}", "macro-F1 natif": f"{block['macro_f1']:.3f}", "précision": with_ci(block["accuracy"], block["accuracy_ci95"])}
    for member in ("mildiou", "alternariose"):
        g = group[member]
        row[f"rappel {member} (n={g['n']})"] = (
            f"exact {with_ci(g['recall_exact'], g['recall_exact_ci95'])} ; groupe {with_ci(g['recall_group'], g['recall_group_ci95'])}"
            if "recall_exact" in g else f"groupe {with_ci(g['recall_group'], g['recall_group_ci95'])}"
        )
    sain = block["per_class"]["sain"]
    row[f"rappel sain (n={sain['n']})"] = with_ci(sain["recall"], sain["recall_ci95"])
    threshold = block.get("threshold", meta["metrics"]["threshold"])
    coverage = block.get("coverage_at_threshold", meta["metrics"]["field_coverage_at_threshold"])
    row["seuil / couverture"] = f"{threshold} / {pct(coverage)}"
    return row


metas = {}
for name in RUN:
    path = WORK / f"out_{name}" / "sillon_disease.json"
    if path.exists():
        metas[name] = json.loads(path.read_text())

for key, title in (("field_test", "Évaluation simple (celle de l'appli)"), ("field_test_tta", "Avec test-time augmentation (indicatif)")):
    print(f"\n#### {title}")
    for name, meta in metas.items():
        print(f"\n{name}  ({meta['backbone']}, {meta['input']['width']} px, recette {meta['recipe']['name']}, fusion : {meta['merge_leaf_spots']})")
        for column, value in summarize(meta, key).items():
            print(f"  {column:28s} {value}")

# Tableau à copier-coller (évaluation simple)
rows = {name: summarize(meta, "field_test") for name, meta in metas.items()}
if rows:
    columns = list(next(iter(rows.values())))
    print("\n| variante | " + " | ".join(columns) + " |")
    print("|" + "---|" * (len(columns) + 1))
    for name, row in rows.items():
        print(f"| {name} | " + " | ".join(row[c] for c in columns) + " |")

In [ ]:
# Critère d'installation : mieux que la baseline sur macro-F1 ET sur le rappel mildiou/alternariose,
# avec des intervalles de confiance qui ne se chevauchent pas trop (borne basse du candidat au moins
# égale à l'estimation de la baseline). Ce n'est qu'un garde-fou : la décision reste humaine.
def block_of(meta):
    return meta["metrics"]["field_test"]


def comparable_f1(meta):
    block = block_of(meta)
    return block["macro_f1"] if meta["merge_leaf_spots"] else block.get("merged_view", {}).get("macro_f1", block["macro_f1"])


def recall_of(meta, member, exact):
    """(rappel, intervalle). exact=True : la maladie nommée ; sinon : rangée dans le groupe des taches de feuille."""
    block = block_of(meta)
    if exact:
        m = block["per_class"][member]
        return m["recall"], m["recall_ci95"]
    g = block["leaf_spot_group"]["members"][member]
    return g["recall_group"], g["recall_group_ci95"]


base = metas.get("baseline_small224")
if not base:
    print("Pas de baseline : comparaison impossible.")
else:
    for name, meta in metas.items():
        if name == "baseline_small224":
            continue
        merged = meta["merge_leaf_spots"]
        f1_ok = comparable_f1(meta) > comparable_f1(base)
        print(f"\n{name} : macro-F1 (12 cl.) {comparable_f1(meta):.3f} contre {comparable_f1(base):.3f} -> {'mieux' if f1_ok else 'PAS mieux'}")
        all_ok = f1_ok
        for member in ("mildiou", "alternariose"):
            # Candidat : rappel exact s'il existe, rappel de groupe s'il est fusionné ; référence : rappel exact de la baseline.
            value, ci = recall_of(meta, member, exact=not merged)
            ref, _ = recall_of(base, member, exact=True)
            credible = ci[0] >= ref and value > ref
            all_ok = all_ok and credible
            kind = "groupe" if merged else "exact"
            print(f"  rappel {member} ({kind}) {pct(value)} [{pct(ci[0])}-{pct(ci[1])}] contre baseline exact {pct(ref)} -> {'amélioration crédible' if credible else 'non démontré'}")
            if not merged:
                gv, gci = recall_of(meta, member, exact=False)
                gref, _ = recall_of(base, member, exact=False)
                print(f"    (en groupe : {pct(gv)} [{pct(gci[0])}-{pct(gci[1])}] contre baseline {pct(gref)})")
        verdict = "candidat à l'installation (relecture humaine requise)" if all_ok else "ne remplace PAS le modèle actuel"
        print(f"  => {verdict}")

## Récupérer les résultats

Dans l'onglet **Output** du notebook (après « Save Version »), téléchargez pour chaque variante `out_<variante>/` :
`sillon_disease.tflite`, `sillon_disease.json`, `rapport_metriques.txt` et `metrics.json`.
**Ne copiez rien dans `app/src/main/assets/models/` vous-même** : renvoyez le tableau comparatif ci-dessus et les
`rapport_metriques.txt`. Un modèle fusionné demande en plus une adaptation de l'appli (voir README, section « Ce que l'appli doit adapter »).

In [ ]:
# Archive : liste des fichiers à télécharger pour chaque variante terminée.
for name in RUN:
    out = WORK / f"out_{name}"
    if (out / "sillon_disease.json").exists():
        keep = [out / f for f in ("sillon_disease.tflite", "sillon_disease.json", "rapport_metriques.txt", "metrics.json")]
        print(name, "->", [f.name for f in keep if f.exists()])